# h 指数の予測力の再現 (OpenAlex) — 書籍 図2.2

Hirsch, *Does the h index have predictive power?* PNAS 104 (2007) / Wang & Barabási *The Science of Science* (2021) 第2章 図2.2。

研究者ごとに、ある時点 $t_1$ で測った 4 つの単一指標が、後の時点 $t_2$ での到達度 $\sqrt{C(t_2)}$ ($t_2$ 時点の累積被引用数の平方根) をどれだけ予測できるかを散布図で比べる。縦軸は 4 枚とも $\sqrt{C(t_2)}$、横軸はそれぞれ:

| パネル | 横軸 | 意味 |
|---|---|---|
| 左上 | $h(t_1)$ | $t_1$ 時点の h 指数 |
| 右上 | $\sqrt{C(t_1)}$ | $t_1$ 時点の累積被引用数の平方根 |
| 左下 | $N(t_1)$ | $t_1$ 時点の論文数 |
| 右下 | $C(t_1)/N(t_1)$ | $t_1$ 時点の論文あたり被引用数 |

Hirsch の主張は「$h(t_1)$ と $\sqrt{C(t_1)}$ は将来 $\sqrt{C(t_2)}$ をよく予測する (相関 $r$ が高い) が、$N(t_1)$ や $C(t_1)/N(t_1)$ は予測力が劣る」。元実験は物理学者 (凝縮系物理) が対象。ここでは同じ実験を **信号処理 (Signal processing) の研究者** で行う。

重い処理 (API取得・キャッシュ・指標復元) はすべて `src/openalex_h_index_prediction.py` 側にある。このノートブックはそれを呼ぶだけ。HTTP クライアント等の土台は `src/openalex_random_impact_rule.py` を再利用する。

In [ ]:
# src/ 内の解析モジュールを読み込む。
from pathlib import Path
import importlib
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'openalex_h_index_prediction.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('リポジトリ直下を特定できません')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import openalex_h_index_prediction as hp
importlib.reload(hp)  # .py を編集したら再読込

In [ ]:
# OpenAlex クライアント。mailto を渡すと polite pool でレートが緩和される。
# 結果は data/cache/openalex_h_index_prediction.json にキャッシュされ、再実行が速くなる。
client = hp.OpenAlexClient(
    mailto="r.ikura@msp-lab.org",
    cache_path=str(ROOT / 'data' / 'cache' / 'openalex_h_index_prediction.json'),
)

## 実行

パラメータはここで調整する。`n_researchers` を増やすほど散布図は安定し相関 $r$ の推定も締まるが、その分 API コール数と時間が増える (研究者1名あたり概ね 2〜3 コール)。

- `window_start=2012`: OpenAlex の `counts_by_year` が遡れる最古の年。**初出版がこの年以降の研究者のみ**を対象にして $t_1$ 時点の被引用を正確に復元する (打ち切り対策)。
- `t1=2019`, `t2=2025`: 予測幅 6 年。`t1` を小さくすると予測幅は広がるが $h(t_1)$ の散らばりが乏しくなる。

In [ ]:
results = hp.run(
    client,
    field_filter="concepts.id:C104267543",  # Signal processing。None で全分野
    n_researchers=150,     # 対象研究者数 (図を安定させるなら 100〜)
    candidate_limit=3000,  # 評価する候補著者の上限 (採択率 ~5% 想定)
    sample_works=10000,    # 候補著者を集めるためサンプリングする論文数
    t1=2019, t2=2025,      # 予測子を測る年 / 到達度を測る年
    # 既定: window_start=2012, min_papers=5, min_record_years=2, 除外国=JP/CN
)
client.save_cache()
results['stats']

### 研究者ごとの内訳

$t_1$ 時点の指標 ($h, C, N, C/N$) と $t_2$ 時点の到達度 ($C, h$) を研究者ごとに一覧する。

In [ ]:
df = pd.DataFrame(results['records'])
cols = ['name', 'first_year', 'last_year', 'n_papers_total',
        'h_t1', 'C_t1', 'N_t1', 'CpN_t1', 'C_t2', 'h_t2']
df[cols].sort_values('C_t2', ascending=False).head(15)

## 可視化

縦軸は 4 枚とも $\sqrt{C(t_2)}$。各パネルに回帰直線と Pearson 相関 $r$ を表示する。$h(t_1)$ と $\sqrt{C(t_1)}$ の $r$ が高く、$N(t_1)$・$C(t_1)/N(t_1)$ の $r$ が低ければ Hirsch の主張を支持する。

In [ ]:
%matplotlib inline
fig = hp.make_figure(results)
fig

## 解釈と注意

### 予測力の順位
相関 $r$ が大きいほど、その指標は将来の被引用 $\sqrt{C(t_2)}$ をよく予測する。Hirsch の元実験では $h(t_1)$ と $\sqrt{C(t_1)}$ の $r$ が高く、$N(t_1)$ と $C(t_1)/N(t_1)$ は低い、という順位が示される。

### `counts_by_year` の 2012 年制約について (打ち切りの正体)

本再現で「初出版が 2012 年以降の研究者のみ」を対象にしているのは、OpenAlex の `counts_by_year` フィールドに起因する制約であり、**信号処理という分野を選んだことによる制約ではない**。この点を実測で確認した内容を以下にまとめる。

**`counts_by_year` とは**: 各論文が持つ「年ごとの被引用数」の内訳。`cited_by_count` (累計) を年単位に分解したもの (0 件の年は配列から省略される)。本モジュールはこれを使って `h(t)`・`C(t)`・`N(t)` を任意の時点 $t$ について論文ごとの追加 API コールなしで復元している。

**実測で確認した事実**:

| 検証対象 | 出版年 | `counts_by_year` の範囲 |
|---|---|---|
| 信号処理 (元の確認) | 1976 | 2012〜2026 |
| 物理学 | 1976 | 2012〜2026 |
| 化学 | 1990 | 2012〜2026 |
| 医学 | 1985 | 2012〜2026 |
| 経済学 | 1970 | 2012〜2026 |

→ **分野を問わず、どんなに古い論文でも `counts_by_year` に 2012 年より前のデータは一切現れない**。2012 年が事実上の下限 (フロア) になっている。これは信号処理特有の現象ではなく OpenAlex 全体の仕様であり、別の分野を選んでいても同じ制約を受けていたはずである。

**公式ドキュメントとの食い違い**: OpenAlex の公式説明は「直近10年分のデータ」だが、これは不正確、または少なくとも紛らわしい。もし文字通り「現在から10年遡るローリングウィンドウ」であれば、今日 (2026年) 時点ではどの論文も一律「2017〜2026年」の範囲になるはずである。しかし実測では:

- 古い論文 (1970〜2000年出版) → 範囲は **2012〜2026年 (15年分)**
- 新しい論文 (2024〜2025年出版) → 範囲の開始年は **論文ごとにバラバラ** (2012年・2016年・2022年・自分の出版年など)

という非一様な挙動が観測された。実態に近い説明は「**2012年より前のデータは原理的に存在しない (固定フロア)。上端は現在まで伸び続ける (年々増加)**」であり、単純な「直近10年」のローリングウィンドウではない。なぜ2012年が下限になっているかを明記した一次資料は見つけられておらず、正確な理由は未確認 (前身データの整備状況等が推測されるが裏は取れていない)。

**本再現への影響**: 上記の制約により、$t_1$ 時点の被引用を正確に復元できるのは初出版 2012 年以降の研究者に限られる。代償として、凝縮系物理のようなベテラン中心の母集団は扱えず、**2012 年以降に始まった比較的若いコホート**での再現になる。元実験 (キャリアの長い物理学者) とは母集団が異なる点に留意。この若いコホート制約のために $h(t_1)$ の値域が狭くなり (実行結果では平均 $h(t_1)$≈3.9)、$h(t_1)$ の予測力が $\sqrt{C(t_1)}$ や $C(t_1)/N(t_1)$ より低く出る一因になっている (詳細は `openalex_h_index_prediction_report.md` §5.2 を参照)。

### その他の注意
- **名寄せ**: OpenAlex の著者同定は日本人名・中国名で不安定 (`docs/notes/openalex_exploration.md` 参照)。既定で JP/CN 系を除外している。異常な総論文数 (> 1000) は同名異人の併合とみなして除外。
- **$C(t_2)$ の定義**: $t_1$ 以降に出した新しい論文も含む全論文の $t_2$ 時点の累積被引用。すなわち「その研究者の将来の総到達度」を予測対象にしている。
- **サンプル**: 候補は 2012〜$t_1$ の信号処理 works から無作為抽出した著者。`n_researchers`/`candidate_limit` を増やすと $r$ の推定が安定する。
